# **Workforce Intelligence Agent**
## **Objective**
This notebook analyzes employee workload, productivity, and capacity to identify operational bottlenecks and generate workforce recommendations. It computes business KPIs that will later be consumed by the Executive Briefing Agent.

## **Database Connection**

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

# Create engine
engine = create_engine(DATABASE_URL)

# Test connection
with engine.connect() as connection:
    print("Successfully connected to the database!")

Successfully connected to the database!


### **Inspect Available Database Tables**

Query the PostgreSQL database to retrieve all tables in the **public** schema and verify the available datasets.

In [2]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public';
"""

pd.read_sql(query, engine)

,table_name
0,projects
1,team
2,time_logs
3,payments
4,clients


### **Load the Team Dataset**

Retrieve all records from the **team** table and preview the first few rows to understand the dataset's structure and contents.

In [3]:
team = pd.read_sql("SELECT * FROM team;", engine)

team.head()

,id,name,role,hourly_rate,capacity,skills
0,14,Savannah Thompson,Backend Developer,74.0,40,"[FastAPI, Django, PostgreSQL, Redis]"
1,15,Carol Mccullough,BI Analyst,80.0,40,"[Tableau, Power BI, SQL, Excel]"
2,16,Jose Allen,Data Engineer,78.0,40,"[Spark, Airflow, PostgreSQL, dbt]"
3,17,Jessica Reynolds,Project Manager,79.0,40,"[Agile, Scrum, Jira, Resource Planning]"
4,18,Andrew Vasquez,Backend Developer,68.0,40,"[FastAPI, Django, PostgreSQL, Redis]"


### **Load the Projects Dataset**

Retrieve all records from the **projects** table and preview the first few rows to understand the dataset's structure and contents.

In [5]:
projects = pd.read_sql("SELECT * FROM projects;", engine)

projects.head()

,id,client_id,name,budget,hours_estimated,hours_logged,deadline,status,margin
0,26,159,Project Reverse-engineered impactful contingency,90457.0,188.0,64.25,2026-09-01,paused,46.54
1,27,26,Project Enterprise-wide client-driven Graphic ...,17081.0,151.0,62.94,2026-02-13,completed,40.25
2,28,87,Project Robust regional Local Area Network,34953.0,184.0,77.27,2026-07-26,active,25.59
3,29,121,Project Exclusive discrete success,118808.0,100.0,68.90,2026-10-12,active,19.94
4,30,146,Project Sharable 6thgeneration utilization,45302.0,79.0,100.20,2026-08-16,active,39.81


### **Load the Time Logs Dataset**

Retrieve all records from the **time_logs** table and preview the first few rows to examine employee time-tracking records.

In [6]:
time_logs = pd.read_sql("SELECT * FROM time_logs;", engine)

time_logs.head()

,id,team_member_id,project_id,log_date,hours_logged,task_status
0,219,34,95,2026-04-30,1.7,blocked
1,220,27,156,2026-06-29,1.4,blocked
2,221,21,147,2026-07-09,5.7,in_progress
3,222,16,142,2026-04-06,5.3,blocked
4,223,39,113,2026-01-29,6.9,completed


### **Inspect the Team Dataset Structure**

Display the data types, non-null values, and overall structure of the **team** dataset to assess data quality and identify potential preprocessing requirements.

In [9]:
team.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           30 non-null     int64  
 1   name         30 non-null     object 
 2   role         30 non-null     object 
 3   hourly_rate  30 non-null     float64
 4   capacity     30 non-null     int64  
 5   skills       30 non-null     object 
dtypes: float64(1), int64(2), object(3)
memory usage: 1.5+ KB


### **Inspect the Projects Dataset Structure**

Display the data types, non-null values, and overall structure of the **projects** dataset to assess data quality and identify potential preprocessing requirements.

In [10]:
projects.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               200 non-null    int64  
 1   client_id        200 non-null    int64  
 2   name             200 non-null    object 
 3   budget           200 non-null    float64
 4   hours_estimated  200 non-null    float64
 5   hours_logged     200 non-null    float64
 6   deadline         200 non-null    object 
 7   status           200 non-null    object 
 8   margin           200 non-null    float64
dtypes: float64(4), int64(2), object(3)
memory usage: 14.2+ KB


### **Inspect the Time Logs Dataset Structure**

Display the data types, non-null values, and overall structure of the **time_logs** dataset to assess data quality and identify potential preprocessing requirements.

In [11]:
time_logs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              2500 non-null   int64  
 1   team_member_id  2500 non-null   int64  
 2   project_id      2500 non-null   int64  
 3   log_date        2500 non-null   object 
 4   hours_logged    2500 non-null   float64
 5   task_status     2500 non-null   object 
dtypes: float64(1), int64(3), object(2)
memory usage: 117.3+ KB


## **Data Pre-processing**

### **Convert Date Columns to Datetime Format**

Convert the **deadline** and **log_date** columns to the datetime data type to enable accurate date-based calculations, filtering, and time-series analysis.

In [19]:
from datetime import timedelta
projects["deadline"] = pd.to_datetime(projects["deadline"])

time_logs["log_date"] = pd.to_datetime(time_logs["log_date"])

### **Filter Recent Activity**
Capacity is measured weekly (40 hours), so workload should only consider the latest week's activity rather than the entire historical dataset.

In [20]:
latest_date = time_logs["log_date"].max()

last_week = latest_date - timedelta(days=7)

recent_logs = time_logs[
    time_logs["log_date"] >= last_week
].copy()

print(f"Latest log date: {latest_date.date()}")
print(f"Analysing data from: {last_week.date()} onwards")

Latest log date: 2026-07-15
Analysing data from: 2026-07-08 onwards


### **Summarize Employee Workload Metrics**

Aggregate the recent time log records to compute key workload metrics for each employee, including total hours worked, number of log entries, completed tasks, blocked tasks, and tasks currently in progress. These metrics provide a comprehensive view of employee productivity and workload.

In [21]:
employee_hours = (
    recent_logs
    .groupby("team_member_id")
    .agg(
        weekly_hours=("hours_logged", "sum"),
        total_logs=("id", "count"),
        completed_tasks=("task_status",
                         lambda x: (x == "completed").sum()),
        blocked_tasks=("task_status",
                       lambda x: (x == "blocked").sum()),
        active_tasks=("task_status",
                      lambda x: (x == "in_progress").sum())
    )
    .reset_index()
)

employee_hours.head()

,team_member_id,weekly_hours,total_logs,completed_tasks,blocked_tasks,active_tasks
0,14,16.6,5,2,1,2
1,15,26.3,7,3,4,0
2,16,16.2,3,3,0,0
3,17,13.5,3,2,0,1
4,18,15.4,3,1,2,0


### **Combine Workload Metrics with Employee Information**

Merge the aggregated workload metrics with the **team** dataset to enrich each employee's record with details such as name, role, and capacity. This creates a unified dataset for analyzing workload distribution and employee performance.

In [22]:
employee_workload = employee_hours.merge(
    team,
    left_on="team_member_id",
    right_on="id",
    how="left"
)

employee_workload.head()

,team_member_id,weekly_hours,total_logs,completed_tasks,blocked_tasks,active_tasks,id,name,role,hourly_rate,capacity,skills
0,14,16.6,5,2,1,2,14,Savannah Thompson,Backend Developer,74.0,40,"[FastAPI, Django, PostgreSQL, Redis]"
1,15,26.3,7,3,4,0,15,Carol Mccullough,BI Analyst,80.0,40,"[Tableau, Power BI, SQL, Excel]"
2,16,16.2,3,3,0,0,16,Jose Allen,Data Engineer,78.0,40,"[Spark, Airflow, PostgreSQL, dbt]"
3,17,13.5,3,2,0,1,17,Jessica Reynolds,Project Manager,79.0,40,"[Agile, Scrum, Jira, Resource Planning]"
4,18,15.4,3,1,2,0,18,Andrew Vasquez,Backend Developer,68.0,40,"[FastAPI, Django, PostgreSQL, Redis]"


### **Calculate Employee Workload and Performance Metrics**

Compute key performance indicators (KPIs) for each employee, including workload utilization, remaining capacity, labour cost, productivity score, and blocked task rate. These metrics provide insights into employee efficiency, workload balance, resource utilization, and potential operational bottlenecks.

In [23]:
# Weekly Utilization (%)
employee_workload["utilization_pct"] = (
    employee_workload["weekly_hours"]
    /
    employee_workload["capacity"]
) * 100

# Remaining Capacity (Hours)
employee_workload["remaining_capacity"] = (
    employee_workload["capacity"]
    -
    employee_workload["weekly_hours"]
)

# Weekly Labour Cost
employee_workload["labour_cost"] = (
    employee_workload["weekly_hours"]
    *
    employee_workload["hourly_rate"]
)

# Productivity Score
employee_workload["productivity_score"] = (
    employee_workload["completed_tasks"]
    /
    employee_workload["total_logs"]
).fillna(0) * 100

# Blocked Task Rate
employee_workload["blocked_rate"] = (
    employee_workload["blocked_tasks"]
    /
    employee_workload["total_logs"]
).fillna(0) * 100

### **Classify Employee Workload Status**

Categorize each employee based on their workload utilization into **Available**, **Optimal**, or **Overloaded**. This classification helps quickly identify underutilized employees, those operating within a healthy workload range, and employees who may be at risk of burnout.

In [24]:
def classify_workload(utilization):

    if utilization >= 95:
        return "Overloaded"

    elif utilization >= 70:
        return "Optimal"

    else:
        return "Available"


employee_workload["status"] = (
    employee_workload["utilization_pct"]
    .apply(classify_workload)
)

## **Section 7: Skills Intelligence**

Transform the employee skills data into a usable format, quantify each employee's skill set, and analyze the overall distribution of skills across the workforce. This provides insights into team capabilities, identifies critical skill gaps, and supports skill-based task allocation and workforce planning.

In [26]:
from collections import Counter

employee_workload["num_skills"] = employee_workload["skills"].apply(len)

skill_counts = Counter()

for skills in employee_workload["skills"]:
    skill_counts.update(skills)

pd.DataFrame(
    skill_counts.items(),
    columns=["Skill", "Employees"]
).sort_values("Employees", ascending=False)

,Skill,Employees
2,PostgreSQL,10
6,SQL,8
0,FastAPI,6
11,Agile,6
14,Resource Planning,6
1,Django,6
12,Scrum,6
13,Jira,6
3,Redis,6
7,Excel,5


## **Intelligent Workload Recommendations**

Identify employees with excessive workloads and recommend suitable team members to redistribute tasks based on their availability and shared technical skills. This approach supports balanced workload allocation while ensuring that tasks are reassigned to employees with relevant expertise, improving productivity and resource utilization.

In [27]:
recommendations = []

for _, employee in employee_workload.iterrows():

    if employee["status"] == "Overloaded":

        available = employee_workload[
            (employee_workload["status"] == "Available") &
            (
                employee_workload["skills"].apply(
                    lambda s: len(set(s).intersection(employee["skills"])) > 0
                )
            )
        ]

        if not available.empty:

            replacement = available.sort_values(
                "remaining_capacity",
                ascending=False
            ).iloc[0]

            recommendations.append({
                "Employee": employee["name"],
                "Issue": "High Workload",
                "Recommendation":
                    f"Move work to {replacement['name']}",
                "Shared Skills":
                    ", ".join(
                        set(employee["skills"])
                        &
                        set(replacement["skills"])
                    )
            })

### **Generate Task Reallocation Recommendations**

Convert the generated recommendations into a structured DataFrame for easy review and analysis. The resulting table highlights overloaded employees, the recommended team members to receive reassigned tasks, and the technical skills they share, enabling informed workload balancing decisions.

In [28]:
recommendations = pd.DataFrame(recommendations)

recommendations

,Employee,Issue,Recommendation,Shared Skills
0,Lisa Mathews,High Workload,Move work to Jessica Reynolds,"Agile, Scrum, Jira, Resource Planning"


### **Create the Employee Workload Dashboard**

Prepare a consolidated dashboard containing key workload, productivity, capacity, cost, and skills metrics for each employee. Sorting the dashboard by workload utilization highlights the most heavily utilized employees, enabling managers to monitor performance, identify bottlenecks, and make informed resource allocation decisions.

In [29]:
dashboard = employee_workload[
    [
        "name",
        "role",
        "weekly_hours",
        "capacity",
        "utilization_pct",
        "remaining_capacity",
        "completed_tasks",
        "blocked_tasks",
        "productivity_score",
        "blocked_rate",
        "labour_cost",
        "num_skills",
        "status"
    ]
].sort_values(
    "utilization_pct",
    ascending=False
)

dashboard.head(10)

,name,role,weekly_hours,capacity,utilization_pct,remaining_capacity,completed_tasks,blocked_tasks,productivity_score,blocked_rate,labour_cost,num_skills,status
29,Lisa Mathews,Project Manager,41.5,40,103.75,-1.5,4,3,44.444444,33.333333,2946.5,4,Overloaded
12,Samantha Harrison,Data Scientist,37.5,40,93.75,2.5,1,4,12.500000,50.000000,3787.5,4,Optimal
27,Scott Stein,Project Manager,37.4,40,93.50,2.6,2,5,25.000000,62.500000,2393.6,4,Optimal
24,Angela Allen,BI Analyst,31.9,40,79.75,8.1,2,3,28.571429,42.857143,2041.6,4,Optimal
9,Larry Haynes,Backend Developer,30.5,40,76.25,9.5,0,0,0.000000,0.000000,2623.0,4,Optimal
14,Virginia Thompson,Project Manager,29.8,40,74.50,10.2,0,3,0.000000,60.000000,1877.4,4,Optimal
20,Lauren Carson,Frontend Developer,29.7,40,74.25,10.3,0,3,0.000000,60.000000,2554.2,4,Optimal
13,Frederick Phillips,Project Manager,28.8,40,72.00,11.2,5,1,71.428571,14.285714,1526.4,4,Optimal
7,Lauren Williams,Data Engineer,27.8,40,69.50,12.2,1,2,16.666667,33.333333,2641.0,4,Available
5,April Roberts,BI Analyst,27.4,40,68.50,12.6,3,1,60.000000,20.000000,1863.2,4,Available


### **Project Health Analysis**

Compare the estimated effort against the actual hours logged for each project to measure project performance. Calculating the effort variance helps identify projects that are under or over budget, enabling better project monitoring, forecasting, and resource planning.

In [31]:
# Compare estimated vs actual project effort

project_health = projects.copy()

project_health["hours_variance"] = (
    project_health["hours_logged"]
    - project_health["hours_estimated"]
)

project_health["variance_pct"] = (
    project_health["hours_variance"]
    / project_health["hours_estimated"]
) * 100

project_health.head()

,id,client_id,name,budget,hours_estimated,hours_logged,deadline,status,margin,hours_variance,variance_pct
0,26,159,Project Reverse-engineered impactful contingency,90457.0,188.0,64.25,2026-09-01,paused,46.54,-123.75,-65.824468
1,27,26,Project Enterprise-wide client-driven Graphic ...,17081.0,151.0,62.94,2026-02-13,completed,40.25,-88.06,-58.317881
2,28,87,Project Robust regional Local Area Network,34953.0,184.0,77.27,2026-07-26,active,25.59,-106.73,-58.005435
3,29,121,Project Exclusive discrete success,118808.0,100.0,68.90,2026-10-12,active,19.94,-31.10,-31.100000
4,30,146,Project Sharable 6thgeneration utilization,45302.0,79.0,100.20,2026-08-16,active,39.81,21.20,26.835443


### **Estimate Final Project Effort**

Estimate the projected total effort for each project by adding any positive effort variance to the hours already logged. This provides an early indication of the final hours required for projects that are currently exceeding their original estimates.

In [32]:
project_health["predicted_final_hours"] = (
    project_health["hours_logged"]
    +
    project_health["hours_variance"].clip(lower=0)
)

project_health.head()

,id,client_id,name,budget,hours_estimated,hours_logged,deadline,status,margin,hours_variance,variance_pct,predicted_final_hours
0,26,159,Project Reverse-engineered impactful contingency,90457.0,188.0,64.25,2026-09-01,paused,46.54,-123.75,-65.824468,64.25
1,27,26,Project Enterprise-wide client-driven Graphic ...,17081.0,151.0,62.94,2026-02-13,completed,40.25,-88.06,-58.317881,62.94
2,28,87,Project Robust regional Local Area Network,34953.0,184.0,77.27,2026-07-26,active,25.59,-106.73,-58.005435,77.27
3,29,121,Project Exclusive discrete success,118808.0,100.0,68.90,2026-10-12,active,19.94,-31.10,-31.100000,68.90
4,30,146,Project Sharable 6thgeneration utilization,45302.0,79.0,100.20,2026-08-16,active,39.81,21.20,26.835443,121.40


### **Assess Project Risk Levels**

Classify each project into **Low**, **Medium**, or **High** risk based on the percentage variance between estimated and actual effort. This risk assessment helps identify projects that may require immediate attention, enabling proactive resource allocation and timely corrective actions.

In [33]:
def classify_project_risk(row):

    if row["variance_pct"] >= 20:
        return "High"

    elif row["variance_pct"] >= 10:
        return "Medium"

    return "Low"


project_health["risk"] = project_health.apply(
    classify_project_risk,
    axis=1
)

project_health.sort_values(
    "variance_pct",
    ascending=False
).head()

,id,client_id,name,budget,hours_estimated,hours_logged,deadline,status,margin,hours_variance,variance_pct,predicted_final_hours,risk
26,52,148,Project Future-proofed explicit focus group,45792.0,68.0,101.30,2026-10-14,active,47.27,33.30,48.970588,134.60,High
85,111,37,Project Visionary explicit throughput,53345.0,171.0,253.12,2026-04-01,completed,33.76,82.12,48.023392,335.24,High
15,41,83,Project Enhanced radical project,78084.0,65.0,95.86,2026-07-31,active,19.50,30.86,47.476923,126.72,High
77,103,83,Project Intuitive high-level website,65905.0,205.0,299.88,2026-03-29,completed,29.79,94.88,46.282927,394.76,High
56,82,58,Project Inverse national flexibility,89458.0,111.0,158.17,2026-07-02,completed,24.56,47.17,42.495495,205.34,High


### **Forecast Next Week's Employee Capacity**

Estimate future employee workload by applying the average project effort overrun to current workload levels. The resulting predicted hours and utilization provide a forward-looking view of resource demand, helping managers anticipate capacity constraints and proactively balance workloads before they become critical.

In [34]:
average_project_overrun = (
    project_health["variance_pct"]
    .clip(lower=0)
    .mean()
)

growth_factor = 1 + (average_project_overrun / 100)

employee_workload["predicted_hours"] = (
    employee_workload["weekly_hours"]
    * growth_factor
)

employee_workload["predicted_utilization"] = (
    employee_workload["predicted_hours"]
    / employee_workload["capacity"]
) * 100

### **Calculate Projected Capacity Gap**

Measure the difference between each employee's predicted workload and their available capacity to identify potential resource shortages. This KPI highlights employees who are expected to exceed their capacity, enabling proactive workload redistribution and informed workforce planning.

In [35]:
employee_workload["capacity_gap"] = (
    employee_workload["predicted_hours"]
    - employee_workload["capacity"]
)

### **Categorize Forecasted Workload Status**

Classify each employee's projected workload into **Healthy**, **Warning**, or **Critical** categories based on their predicted utilization. This forecast enables managers to identify employees at risk of exceeding their capacity and take proactive actions to prevent resource constraints and maintain project delivery.

In [36]:
def forecast_status(utilization):

    if utilization >= 100:
        return "Critical"

    elif utilization >= 90:
        return "Warning"

    return "Healthy"


employee_workload["forecast_status"] = (
    employee_workload["predicted_utilization"]
    .apply(forecast_status)
)

### **Intelligent Resource Allocation**

In [37]:
future_recommendations = []

for _, overloaded in employee_workload.iterrows():

    if overloaded["forecast_status"] == "Critical":

        candidates = employee_workload[
            (employee_workload["forecast_status"] == "Healthy") &
            (
                employee_workload["skills"].apply(
                    lambda s: len(
                        set(s).intersection(overloaded["skills"])
                    ) > 0
                )
            )
        ]

        if len(candidates):

            replacement = candidates.sort_values(
                "remaining_capacity",
                ascending=False
            ).iloc[0]

            shared = list(
                set(overloaded["skills"])
                &
                set(replacement["skills"])
            )

            future_recommendations.append({

                "Overloaded Employee":
                    overloaded["name"],

                "Predicted Utilization":
                    round(
                        overloaded["predicted_utilization"],
                        1
                    ),

                "Suggested Employee":
                    replacement["name"],

                "Available Capacity":
                    round(
                        replacement["remaining_capacity"],
                        1
                    ),

                "Shared Skills":
                    ", ".join(shared)

            })

future_recommendations = pd.DataFrame(
    future_recommendations
)

future_recommendations

,Overloaded Employee,Predicted Utilization,Suggested Employee,Available Capacity,Shared Skills
0,Lisa Mathews,108.2,Jessica Reynolds,26.5,"Agile, Scrum, Jira, Resource Planning"


### **Analyze Employee Project Assignments**

Aggregate the hours each employee has contributed to individual projects and combine the results with project information such as name, deadline, and status. This creates a comprehensive view of employee project allocations, supporting workload analysis, project tracking, and resource planning.

In [38]:
project_assignment = (
    recent_logs
    .groupby(["team_member_id", "project_id"])
    .agg(
        hours=("hours_logged", "sum")
    )
    .reset_index()
)

project_assignment = (
    project_assignment
    .merge(
        projects[
            [
                "id",
                "name",
                "deadline",
                "status"
            ]
        ],
        left_on="project_id",
        right_on="id",
        how="left"
    )
)

project_assignment.head()

,team_member_id,project_id,hours,id,name,deadline,status
0,14,86,5.5,86,Project Vision-oriented intangible synergy,2026-08-10,active
1,14,92,3.9,92,Project Persevering coherent archive,2026-02-01,completed
2,14,111,2.7,111,Project Visionary explicit throughput,2026-04-01,completed
3,14,117,3.0,117,Project User-centric intermediate database,2026-09-11,paused
4,14,202,1.5,202,Project Organized asynchronous infrastructure,2026-10-01,active


### **Generate Executive Workload Alerts**

Create executive-level alerts that identify employees projected to exceed their capacity, highlight the projects driving the increased workload, and recommend suitable employees for task reallocation based on shared technical skills. These alerts provide concise, actionable insights to support proactive workforce and project management.

In [39]:
executive_alerts = []

for _, row in future_recommendations.iterrows():

    employee_projects = project_assignment[
        project_assignment["team_member_id"] ==
        employee_workload.loc[
            employee_workload["name"] ==
            row["Overloaded Employee"],
            "team_member_id"
        ].values[0]
    ]

    top_project = (
        employee_projects
        .sort_values("hours", ascending=False)
        .iloc[0]
    )

    executive_alerts.append({

        "Alert":
        (
            f"{row['Overloaded Employee']} is projected to reach "
            f"{row['Predicted Utilization']:.0f}% capacity next week "
            f"due to '{top_project['name']}'. "
            f"Recommend reallocating work to "
            f"{row['Suggested Employee']} "
            f"who shares "
            f"{row['Shared Skills']}."
        )

    })

executive_alerts = pd.DataFrame(executive_alerts)

executive_alerts

,Alert
0,Lisa Mathews is projected to reach 108% capaci...
